Esercizio 1 
Genera un dataset di 500.000 righe con colonne id, name, salary e departement. Carica solo le colonne necessarie, elimina duplicati e valori nulli, applica anonimizzazione su name e imposta id come indice. Misura la memoria e i tempi di ricerca prima e dopo le ottimizzazioni.


In [8]:
import pandas as pd 
import numpy as np 
import hashlib 
import time 

dipartimenti = [
    "Risorse Umane", 
    "Contabilità", 
    "Sviluppo Software", 
    "Marketing", 
    "Vendite", 
    "Logistica", 
    "Ricerca e Sviluppo",
    "Customer Care"
]

# Generazione di un dataset generico 
df = pd.DataFrame({
    "id": np.arange(1,500001, dtype=np.int64),
    "name": np.random.choice(["Alice","Bob","Carla","David","Alessia","Daniele","Roberto","Alessandra"], size=500000),
    "salary": np.random.uniform(1000,5000, size = 500000),
    "departement": np.random.choice(dipartimenti, size= 500000)
})

# Salvataggio su disco in formato CSV senza salvare l'indice predefinito di Pandas 
df.to_csv("data.csv",index = False) 

# Caricamento dei dati specificando le colonne da leggere 
df = pd.read_csv("data.csv", usecols=["id","name","salary","departement"])

# Pulizia 
df = df.drop_duplicates() # rimuove eventuali righe identiche 
df = df.dropna(subset=["salary"]) #rimuove le righe con valori nulli/mancanti nella colonna "salary"

print("Ricerca per ID = 5000 prima dell'indice. ")
start_time = time.perf_counter()
risultato_prima = df[df["id"] == 5000] # senza indice si usa la maschera booleana
end_time = time.perf_counter()

print(risultato_prima)
print(f"tempo prima dell'indice: {(end_time - start_time):.4}ms")


# Imposto ID come indice 
df = df.set_index("id")

start_time = time.perf_counter()
risultato_dopo = df.loc[5000]
end_time = time.perf_counter()

print(risultato_dopo)
print(f"Tempo dopo l'indice: {(end_time - start_time):.4}ms")


# Applico la pseudonimizzazione 
salt = "mysalt123"

# Generazione del codice hash SHA-256 per ciascun nome 
# il nome viene unito al salt, codificato in byte e trasformato in stringa esadecimale

df["name_hmac"] = df["name"].map(
    lambda x : hashlib.sha256((salt + str(x)).encode()).hexdigest() if pd.notna(x) else None
)

print("Consumo iniziale di memoria (MB):")
print(df.memory_usage(deep=True).sum() /1024**2)

df.index = df.index.astype("int32")
df["name"] = df["name"].astype("category")
df["salary"] = df["salary"].astype("float")
df["departement"] = df["departement"].astype("category")

print("Consumo di memoria dopo il downcasting (MB):")
print(df.memory_usage(deep=True).sum() /1024**2)



Ricerca per ID = 5000 prima dell'indice. 
        id   name       salary    departement
4999  5000  Carla  1300.075312  Customer Care
tempo prima dell'indice: 0.0009534ms
name                   Carla
salary           1300.075312
departement    Customer Care
Name: 5000, dtype: object
Tempo dopo l'indice: 0.01067ms
Consumo iniziale di memoria (MB):
134.80689811706543
Consumo di memoria dopo il downcasting (MB):
60.55980587005615


Esercizio 2 
Crea un dataset con 1 milione di righe, includendo colonne numeriche con valori grandi. Applica downcasting a int32 e float32 e confronta l’uso della memoria. Poi esegui un’aggregazione complessa (group by e sum) e confronta i tempi prima e dopo il downcasting.


In [7]:
import numpy as np
import pandas as pd
import time 

# Definizione del numero di righe
num_rows = 1_000_000

# Impostazione del seed per la riproducibilità
np.random.seed(42)

# Generazione delle colonne numeriche grandi
id_dipendente = np.arange(1, num_rows + 1, dtype=np.int64)
transazione_id = np.random.randint(
    1, 9_999_999, size=num_rows, dtype=np.int64
)

# Generazione delle colonne stringa
nomi = [
    "Alice",
    "Bob",
    "Carla",
    "David",
    "Alessia",
    "Daniele",
    "Roberto",
    "Alessandra",
    "Elena",
    "Marco",
]
dipartimenti = [
    "Risorse Umane",
    "Contabilità",
    "Sviluppo Software",
    "Marketing",
    "Vendite",
    "Logistica",
    "R&D",
    "Customer Care",
]

col_nomi = np.random.choice(nomi, size=num_rows)
col_dipartimenti = np.random.choice(dipartimenti, size=num_rows)

# Valori monetari float
stipendio_annuale = np.round(
    np.random.uniform(25000.00, 150000.00, size=num_rows), 2
)
budget_gestito = np.round(
    np.random.uniform(1_000_000.00, 500_000_000.00, size=num_rows), 2
)

# Creazione del DataFrame
df = pd.DataFrame({
    "ID_Dipendente": id_dipendente,
    "ID_Transazione": transazione_id,
    "Nome": col_nomi,
    "Dipartimento": col_dipartimenti,
    "Stipendio_Annuale_EUR": stipendio_annuale,
    "Budget_Gestito_EUR": budget_gestito,
})

# Mantiene i dati numerici e rimuove la notazione scientifica dalla visualizzazione
pd.options.display.float_format = "{:.2f}".format
print(df.head(10))

# Downcasting e uso della memoria 

# TEMPI E MEMORIA PRIMA DEL DOWNCASTING 

mem_prima = df.memory_usage(deep =True).sum() / 1024**2 # calcolo la memoria usata
start = time.perf_counter() # tempo iniziale
res_prima = df.groupby(["Dipartimento", "Nome"])[
    ["Stipendio_Annuale_EUR", "Budget_Gestito_EUR"]
].sum()
tempo_prima = time.perf_counter() - start


# APPLICAZIONE DEL CASTING OTTIMIZZATO

df["ID_Dipendente"] = pd.to_numeric(df["ID_Dipendente"], downcast="integer")
df["ID_Transazione"] = pd.to_numeric(df["ID_Transazione"], downcast="integer")
df["Nome"] = df["Nome"].astype("category")
df["Dipartimento"] = df["Dipartimento"].astype("category")
df["Stipendio_Annuale_EUR"] = df["Stipendio_Annuale_EUR"].astype("float32")
df["Budget_Gestito_EUR"] = df["Budget_Gestito_EUR"].astype("float32")

# TEMPI E MEMORIA DOPO IL DOWNCASTING 

mem_dopo = df.memory_usage(deep=True).sum() / 1014**2
start = time.perf_counter()
res_dopo = df.groupby(["Dipartimento","Nome"], observed= False)[
    ["Stipendio_Annuale_EUR", "Budget_Gestito_EUR"]
].sum()
tempo_dopo = time.perf_counter() - start


print("--- RISULTATI ---")
print(f"Memoria RAM:  {mem_prima:.2f} MB  -->  {mem_dopo:.2f} MB  (Riduzione del {((mem_prima - mem_dopo)/mem_prima)*100:.1f}%)")
print(f"Tempo GroupBy:{tempo_prima*1000:.2f} ms -->  {tempo_dopo*1000:.2f} ms (Velocizzazione di {tempo_prima/tempo_dopo:.2f}x)")


   ID_Dipendente  ID_Transazione     Nome       Dipartimento  \
0              1         6423389    Elena  Sviluppo Software   
1              2         6550635  Roberto  Sviluppo Software   
2              3         4304573    Elena      Customer Care   
3              4         2234490    Carla            Vendite   
4              5         9958615  Alessia                R&D   
5              6         9524683    Elena      Risorse Umane   
6              7         7204213    Marco      Risorse Umane   
7              8         9628520    Elena          Logistica   
8              9         4472472    Elena            Vendite   
9             10         4523670    Marco        Contabilità   

   Stipendio_Annuale_EUR  Budget_Gestito_EUR  
0               25035.72        115440990.90  
1               80705.13         50694838.07  
2               86684.27        327334869.63  
3              132849.60        438176076.66  
4               41189.82        476045221.66  
5            

Esercizio 3
Genera un dataset misto con colonne id, name, salary, city e departement, con valori ripetuti e mancanti casuali. Applica pulizia, anonimizzazione dei nomi, indicizzazione avanzata e conversione di colonne categoriche.


In [8]:
import pandas as pd
import numpy as np
import random
import hashlib

# ---------------------------------------------------------
# 1. GENERAZIONE DEL DATASET MISTO (con ripetizioni e NaN)
# ---------------------------------------------------------
np.random.seed(42)
random.seed(42)

n_rows = 20

nomi = ["Marco", "Giulia", "Luca", "Anna", "Matteo", "Sara", None]
citta = ["Milano", "Roma", "Napoli", "Torino", None]
dipartimenti = ["IT", "HR", "Finance", "Marketing", None]

data = {
    "id": np.random.choice([101, 102, 103, 104, 105, 106, 107, 108], size=n_rows),
    "name": [random.choice(nomi) for _ in range(n_rows)],
    "salary": np.random.choice([25000.0, 32000.0, 45000.0, 50000.0, np.nan], size=n_rows),
    "city": [random.choice(citta) for _ in range(n_rows)],
    "departement": [random.choice(dipartimenti) for _ in range(n_rows)]
}

df = pd.DataFrame(data)
print("=== 1. DATASET ORIGINALE ===")
print(df.head(10))
print("\n" + "="*50 + "\n")


# ---------------------------------------------------------
# 2. PULIZIA DEI DATI
# ---------------------------------------------------------
# A. RIMOZIONE DUPLICATI: Elimina righe completamente identiche
df = df.drop_duplicates().reset_index(drop=True)

# B. GESTIONE VALORI MANCANTI (NaN)
# - Per 'salary': imputazione con la media
df["salary"] = df["salary"].fillna(df["salary"].mean())

# - Per le colonne di testo: imputazione con 'Sconosciuto' / 'Non Assegnato'
df["name"] = df["name"].fillna("Anonimo")
df["city"] = df["city"].fillna("Sconosciuta")
df["departement"] = df["departement"].fillna("Non Assegnato")

print("=== 2. DATASET PULITO ===")
print(df.head(10))
print("\n" + "="*50 + "\n")


# ---------------------------------------------------------
# 3. ANONIMIZZAZIONE DEI NOMI (Hashing SHA-256)
# ---------------------------------------------------------
# Sostituisce il nome reale con i primi 8 caratteri dell'hash del nome
def anonimizza_nome(valore):
    return hashlib.sha256(valore.encode()).hexdigest()[:8]

df["name"] = df["name"].apply(anonimizza_nome)

print("=== 3. NOMI ANONIMIZZATI ===")
print(df[["id", "name"]].head(10))
print("\n" + "="*50 + "\n")


# ---------------------------------------------------------
# 4. CONVERSIONE A TIPI CATEGORICI ED EFFICIENZA
# ---------------------------------------------------------
# Conversione delle colonne a bassa cardinalità in 'category'
df["city"] = df["city"].astype("category")
df["departement"] = df["departement"].astype("category")

# Downcasting numerico
df["id"] = pd.to_numeric(df["id"], downcast="integer")
df["salary"] = df["salary"].astype("float32")

print("=== 4. TIPI DI DATO E MEMORIA ===")
print(df.dtypes)
print("\n" + "="*50 + "\n")


# ---------------------------------------------------------
# 5. INDICIZZAZIONE AVANZATA (MultiIndex)
# ---------------------------------------------------------
# Creazione di un indice gerarchico su 'departement' e 'city'
df_indexed = df.set_index(["departement", "city"]).sort_index()

print("=== 5. DATASET CON MULTIINDEX ===")
print(df_indexed.head(10))

# Esempio di selezione avanzata con il MultiIndex
print("\n--- Esempio di Query su MultiIndex (Dipartimento: IT) ---")
print(df_indexed.loc["IT"])

=== 1. DATASET ORIGINALE ===
    id    name   salary    city departement
0  107    Sara      NaN    Roma          IT
1  104   Marco 32000.00    None          IT
2  105   Marco 50000.00    None   Marketing
3  107    Sara 32000.00  Milano          IT
4  103    Luca 50000.00    None     Finance
5  108  Giulia      NaN    Roma     Finance
6  105  Giulia 25000.00    None        None
7  105  Giulia 50000.00  Torino     Finance
8  107    Sara 32000.00    Roma          IT
9  102   Marco      NaN  Torino   Marketing


=== 2. DATASET PULITO ===
    id    name   salary         city    departement
0  107    Sara 40470.59         Roma             IT
1  104   Marco 32000.00  Sconosciuta             IT
2  105   Marco 50000.00  Sconosciuta      Marketing
3  107    Sara 32000.00       Milano             IT
4  103    Luca 50000.00  Sconosciuta        Finance
5  108  Giulia 40470.59         Roma        Finance
6  105  Giulia 25000.00  Sconosciuta  Non Assegnato
7  105  Giulia 50000.00       Torino       